# Vision Mamba — Dyslexic Handwriting Detection

Trained on the real **Dyslexia Handwriting Dataset** (Isa et al., Kaggle: `drizasazanitaisa/dyslexia-handwriting-dataset`),
which you've confirmed has this structure:

```
dyslexic/train/normal/*
dyslexic/train/reversal/*
dyslexic/train/corrected/*
dyslexic/test/normal/*
dyslexic/test/reversal/*
dyslexic/test/corrected/*
```

**Label mapping** (this is the real signal — same collection pipeline for all three classes, no domain-mismatch shortcut):
- `normal` → **Not-Dyslexic** (label 0)
- `reversal` + `corrected` → **Dyslexic** (label 1)

There's no `valid` folder, so we carve a stratified validation split out of `train`.

**Workflow:** upload your dataset zip straight from your laptop (no Drive mount, no Kaggle API needed here since
you already have the file) → train a Vision Mamba (bidirectional selective-SSM) binary classifier → evaluate
→ per-class (normal/reversal/corrected) accuracy breakdown → predict on new images/folders.

> Vision Mamba is implemented in pure PyTorch (no `mamba-ssm`/`causal-conv1d` CUDA kernels needed), so it runs
> on a stock Colab GPU runtime (T4/A100) with no compilation step.


## 0. Setup

In [ ]:
!pip -q install einops scikit-learn pillow tqdm
import os, glob, json, random, re, zipfile, shutil
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cpu


## 1. Upload your dataset zip

Click "Choose Files" below and pick the zip from your laptop. It will be extracted to `/content/dyslexic_dataset`.
If your zip has an extra wrapper folder (e.g. `dyslexic/train/...` vs `train/...` directly), the auto-detect
step right after will find it either way.

In [ ]:
from google.colab import files
print('Select your dataset zip file...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
EXTRACT_ROOT = '/content/dyslexic_dataset'
if os.path.exists(EXTRACT_ROOT):
    shutil.rmtree(EXTRACT_ROOT)
os.makedirs(EXTRACT_ROOT, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(EXTRACT_ROOT)

print('Extracted. Top-level contents:')
print(os.listdir(EXTRACT_ROOT))


Select your dataset zip file...


Saving dyslexic.zip to dyslexic.zip
Extracted. Top-level contents:
['dyslexic']


In [ ]:
# Auto-detect the real root: the folder that directly contains 'train' and 'test'
EXTRACT_ROOT = '/content/dyslexic_dataset'
def find_dataset_root(base):
    for dirpath, dirnames, _ in os.walk(base):
        lowered = [d.lower() for d in dirnames]
        if 'train' in lowered and 'test' in lowered:
            return dirpath
    return None

DATA_ROOT = find_dataset_root(EXTRACT_ROOT)
assert DATA_ROOT is not None, 'Could not find train/test folders — check the zip structure printed above.'
print('Detected dataset root:', DATA_ROOT)

CLASS_DIRS = {}  # actual-case folder names, e.g. {'train': {'normal': 'Normal', ...}, ...}
for split in ['train', 'test']:
    split_path = os.path.join(DATA_ROOT, split)
    # handle case-insensitive match for the split folder itself
    real_split = next(d for d in os.listdir(DATA_ROOT) if d.lower() == split)
    split_path = os.path.join(DATA_ROOT, real_split)
    classes = {d.lower(): d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))}
    CLASS_DIRS[split] = {'root': split_path, 'classes': classes}
    print(f"{split}: found classes {list(classes.values())}")
    for lower, real in classes.items():
        n = len(glob.glob(os.path.join(split_path, real, '*')))
        print(f'   {real}: {n} files')


Detected dataset root: /content/dyslexic_dataset/dyslexic
train: found classes ['Corrected', 'Reversal', 'Normal']
   Corrected: 65534 files
   Reversal: 46781 files
   Normal: 39334 files
test: found classes ['Corrected', 'Reversal', 'Normal']
   Corrected: 19284 files
   Reversal: 17882 files
   Normal: 19557 files


## 2. Build the manifest (binary label + original 3-way class kept for later breakdown)

We also try to pull a leading letter (A-Z) out of each filename, purely as a best-effort convenience for the
per-letter inspection helper later — if filenames don't encode the letter, that column will just say `unknown`
and per-letter inspection won't be meaningful (binary training/evaluation is unaffected either way).

In [ ]:
LABEL_MAP = {'normal': 0, 'reversal': 1, 'corrected': 1}   # 0 = Not-Dyslexic, 1 = Dyslexic
CLASS_NAMES = ['Not-Dyslexic', 'Dyslexic']

LETTER_RE = re.compile(r'^([A-Za-z])[_\-\.]')

def guess_letter(filename):
    m = LETTER_RE.match(os.path.basename(filename))
    return m.group(1).upper() if m else 'unknown'

rows = []
for split, info in CLASS_DIRS.items():
    for lower_cls, real_cls in info['classes'].items():
        if lower_cls not in LABEL_MAP:
            print(f'WARNING: unexpected class folder {real_cls!r} in {split} — skipping')
            continue
        label = LABEL_MAP[lower_cls]
        for fp in glob.glob(os.path.join(info['root'], real_cls, '*')):
            rows.append({
                'path': fp, 'class3': lower_cls, 'label': label,
                'letter': guess_letter(fp), 'orig_split': split,
            })

full_manifest = pd.DataFrame(rows)
print('Total images:', len(full_manifest))
print(full_manifest.groupby(['orig_split', 'class3']).size())


Total images: 208372
orig_split  class3   
test        corrected    19284
            normal       19557
            reversal     17882
train       corrected    65534
            normal       39334
            reversal     46781
dtype: int64


In [ ]:
# Carve train -> train/valid (stratified on the 3-way class so normal/reversal/corrected
# proportions stay consistent), keep the original test set untouched as the final held-out set.
train_pool = full_manifest[full_manifest['orig_split'] == 'train'].copy()
test_df    = full_manifest[full_manifest['orig_split'] == 'test'].copy()

train_df, valid_df = train_test_split(
    train_pool, test_size=0.15, stratify=train_pool['class3'], random_state=SEED
)
train_df['split'] = 'train'
valid_df['split'] = 'valid'
test_df['split']  = 'test'

manifest = pd.concat([train_df, valid_df, test_df], ignore_index=True)
manifest.to_csv('/content/manifest.csv', index=False)

print(manifest.groupby(['split', 'label']).size())
print()
print(manifest.groupby(['split', 'class3']).size())


split  label
test   0        19557
       1        37166
train  0        33434
       1        95467
valid  0         5900
       1        16848
dtype: int64

split  class3   
test   corrected    19284
       normal       19557
       reversal     17882
train  corrected    55703
       normal       33434
       reversal     39764
valid  corrected     9831
       normal        5900
       reversal      7017
dtype: int64


## 3. Dataset / DataLoader

In [ ]:
# Peek at a real sample to see native resolution/mode before choosing IMG_SIZE
_sample_path = manifest.iloc[0]['path']
_sample_img = Image.open(_sample_path)
print('Sample image:', _sample_path)
print('Native size/mode:', _sample_img.size, _sample_img.mode)


Sample image: /content/dyslexic_dataset/dyslexic/Train/Corrected/6_3103.png
Native size/mode: (30, 30) L


In [ ]:
IMG_SIZE = 64          # resize target regardless of native resolution
PATCH_SIZE = 8         # -> (64/8)^2 = 64 patch tokens
IN_CHANS = 1           # grayscale

class HandwritingDataset(Dataset):
    def __init__(self, df, img_size=IMG_SIZE):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('L').resize((self.img_size, self.img_size))
        x = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).unsqueeze(0)
        x = (x - 0.5) / 0.5
        return x, int(row['label'])

train_ds = HandwritingDataset(manifest[manifest['split'] == 'train'])
valid_ds = HandwritingDataset(manifest[manifest['split'] == 'valid'])
test_ds  = HandwritingDataset(manifest[manifest['split'] == 'test'])

train_labels = train_ds.df['label'].values
class_counts = np.bincount(train_labels, minlength=2)
class_weights = 1.0 / np.clip(class_counts, 1, None)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = 256
NUM_WORKERS = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

print('train/valid/test sizes:', len(train_ds), len(valid_ds), len(test_ds))
print('train class counts (Not-Dyslexic, Dyslexic):', class_counts)


train/valid/test sizes: 128901 22748 56723
train class counts (Not-Dyslexic, Dyslexic): [33434 95467]


## 4. Vision Mamba model (pure PyTorch, bidirectional selective SSM)

- **PatchEmbed**: conv2d patchifies the image into a token sequence + learned positional embedding.
- **MambaBlock**: pre-norm residual block — input-dependent `(Δ, B, C)`, diagonal state `A`, discretized and
  scanned **forward and backward** over the patch sequence (images have no natural causal order), gated with
  `SiLU`.
- Stack of blocks → mean-pool over tokens → linear head → 2 logits.

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=IMG_SIZE, patch_size=PATCH_SIZE, in_chans=IN_CHANS, embed_dim=96):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        n_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x + self.pos_embed


class SelectiveSSM(nn.Module):
    def __init__(self, d_inner, d_state=16):
        super().__init__()
        self.d_inner = d_inner
        self.d_state = d_state
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.x_proj = nn.Linear(d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, d_inner, bias=True)

    def forward(self, x, reverse=False):
        B_, L, D_ = x.shape
        if reverse:
            x = x.flip(1)

        A = -torch.exp(self.A_log)
        x_dbl = self.x_proj(x)
        Bp, Cp, delta = torch.split(x_dbl, [self.d_state, self.d_state, 1], dim=-1)
        delta = F.softplus(self.dt_proj(delta))

        dA = torch.exp(delta.unsqueeze(-1) * A)
        dB = delta.unsqueeze(-1) * Bp.unsqueeze(2)

        h = x.new_zeros(B_, D_, self.d_state)
        ys = []
        for t in range(L):
            h = dA[:, t] * h + dB[:, t] * x[:, t].unsqueeze(-1)
            y_t = (h * Cp[:, t].unsqueeze(1)).sum(-1)
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        y = y + x * self.D

        if reverse:
            y = y.flip(1)
        return y


class MambaBlock(nn.Module):
    def __init__(self, d_model, expand=2, d_state=16, conv_kernel=3):
        super().__init__()
        d_inner = d_model * expand
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, d_inner * 2)
        self.conv = nn.Conv1d(d_inner, d_inner, kernel_size=conv_kernel,
                               padding=conv_kernel - 1, groups=d_inner)
        self.ssm_fwd = SelectiveSSM(d_inner, d_state)
        self.ssm_bwd = SelectiveSSM(d_inner, d_state)
        self.out_proj = nn.Linear(d_inner, d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        x, z = self.in_proj(x).chunk(2, dim=-1)

        x_conv = self.conv(x.transpose(1, 2))[..., :x.shape[1]].transpose(1, 2)
        x_conv = F.silu(x_conv)

        y = self.ssm_fwd(x_conv, reverse=False) + self.ssm_bwd(x_conv, reverse=True)
        y = y * F.silu(z)
        y = self.out_proj(y)
        return residual + y


class VisionMamba(nn.Module):
    def __init__(self, img_size=IMG_SIZE, patch_size=PATCH_SIZE, in_chans=IN_CHANS,
                 embed_dim=96, depth=4, d_state=16, num_classes=2, drop_rate=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        self.pos_drop = nn.Dropout(drop_rate)
        self.blocks = nn.ModuleList([MambaBlock(embed_dim, d_state=d_state) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.head(x)

model = VisionMamba().to(DEVICE)
print(f'VisionMamba parameters: {sum(p.numel() for p in model.parameters()):,}')


VisionMamba parameters: 319,586


## 5. Train

In [ ]:
!pip install -q --upgrade "sympy>=1.13.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 54.9 MB/s eta 0:00:00


In [ ]:
EPOCHS = 20
LR = 3e-4
PATIENCE = 5

# Use Adam instead of AdamW to avoid the SymPy/PyTorch AdamW initialization issue
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

criterion = nn.CrossEntropyLoss()

# AMP
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

CKPT_PATH = '/content/vision_mamba_dyslexia_best.pt'
best_val_loss = float('inf')
patience_ct = 0


def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    n = 0

    with torch.set_grad_enabled(train):

        for x, y in tqdm(loader, leave=False):

            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=(DEVICE.type == "cuda")
            ):
                logits = model(x)
                loss = criterion(logits, y)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            n += x.size(0)

    return total_loss / n, correct / n


for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc = run_epoch(
        train_loader,
        train=True
    )

    val_loss, val_acc = run_epoch(
        valid_loader,
        train=False
    )

    scheduler.step()

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_ct = 0

        torch.save(
            {
                'model_state': model.state_dict(),
                'img_size': IMG_SIZE,
                'patch_size': PATCH_SIZE
            },
            CKPT_PATH
        )

    else:
        patience_ct += 1

        if patience_ct >= PATIENCE:
            print("Early stopping.")
            break


print(
    "Best val loss:",
    best_val_loss,
    "-> checkpoint saved at",
    CKPT_PATH
)

  0%|          | 0/504 [00:00<?, ?it/s]

  0%|          | 0/89 [00:00<?, ?it/s]

Epoch 01 | train loss 0.3227 acc 0.8632 | val loss 0.2251 acc 0.9113


  0%|          | 0/504 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6c34a50680>Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6c34a50680>

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
if w.is_alive():    self._shutdown_workers()
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
assert self._parent_pid == os.getpid(), 'can only test a child process'
    AssertionErrorif w.is_alive():: 
can only test a child process  File "/usr/lib/p

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6c34a50680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6c34a50680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

  0%|          | 0/89 [00:00<?, ?it/s]

Epoch 06 | train loss 0.0674 acc 0.9771 | val loss 0.0935 acc 0.9676


  0%|          | 0/504 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6c34a50680>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7e6c34a50680>    self._shutdown_workers()
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__

      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()    
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
if w.is_alive():
    if w.is_alive():  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    
assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3.13/multiprocessing/process.py", line 16

## 6. Test-set evaluation (overall + per original class breakdown)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

test_df_eval = manifest[manifest['split'] == 'test'].reset_index(drop=True)
all_preds, all_probs = [], []
with torch.no_grad():
    for i in range(0, len(test_df_eval), BATCH_SIZE):
        batch_rows = test_df_eval.iloc[i:i + BATCH_SIZE]
        imgs = torch.stack([
            (torch.from_numpy(np.array(Image.open(p).convert('L').resize((IMG_SIZE, IMG_SIZE)),
                                        dtype=np.float32) / 255.0).unsqueeze(0) - 0.5) / 0.5
            for p in batch_rows['path']
        ]).to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)
        all_preds.extend(probs.argmax(1).cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

test_df_eval['pred'] = all_preds
test_df_eval['dyslexic_prob'] = all_probs

print('=== Overall binary performance ===')
print(classification_report(test_df_eval['label'], test_df_eval['pred'], target_names=CLASS_NAMES))
print('Confusion matrix [rows=true, cols=pred]:')
print(confusion_matrix(test_df_eval['label'], test_df_eval['pred']))

print()
print('=== Accuracy by original 3-way class (normal / reversal / corrected) ===')
for cls in ['normal', 'reversal', 'corrected']:
    sub = test_df_eval[test_df_eval['class3'] == cls]
    if len(sub) == 0:
        continue
    acc = (sub['pred'] == sub['label']).mean()
    print(f'{cls:10s}: {acc:.2%}  (n={len(sub)})')


## 7. Inference helpers

- `predict_image(path)` — classify a single image file.
- `predict_folder(folder)` — classify every image in a folder (e.g. a folder of a specific letter, if you have
  per-letter folders elsewhere, or any ad hoc set of images you want screened).
- `predict_letter(letter, split='test')` — only meaningful if filenames encoded a leading letter (check the
  `letter` column in `manifest` — if it's mostly `'unknown'`, this dataset's filenames don't carry letter info
  and this helper won't have anything to filter on).

In [ ]:
def _load_image(path, img_size=IMG_SIZE):
    img = Image.open(path).convert('L').resize((img_size, img_size))
    x = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).unsqueeze(0)
    return (x - 0.5) / 0.5

def predict_image(path):
    model.eval()
    x = _load_image(path).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0]
    pred = probs.argmax().item()
    return CLASS_NAMES[pred], probs[pred].item()

def predict_folder(folder, show=20):
    paths = sorted(glob.glob(os.path.join(folder, '*')))
    if not paths:
        print(f'No images found in {folder}')
        return
    model.eval()
    with torch.no_grad():
        for p in paths[:show]:
            x = _load_image(p).unsqueeze(0).to(DEVICE)
            probs = F.softmax(model(x), dim=1)[0]
            pred = probs.argmax().item()
            print(f'  {os.path.basename(p):30s} -> {CLASS_NAMES[pred]:13s} ({probs[pred].item():.2%})')
    if len(paths) > show:
        print(f'  ... and {len(paths) - show} more')

def predict_letter(letter, split='test', show=20):
    letter = letter.upper()
    sub = manifest[(manifest['letter'] == letter) & (manifest['split'] == split)]
    if sub.empty:
        print(f"No images with detected letter '{letter}' in split '{split}' "
              f"(letters may not be encoded in these filenames — check manifest['letter'].value_counts()).")
        return
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for i, (_, row) in enumerate(sub.iterrows()):
            x = _load_image(row['path']).unsqueeze(0).to(DEVICE)
            probs = F.softmax(model(x), dim=1)[0]
            pred = probs.argmax().item()
            total += 1
            correct += int(pred == row['label'])
            if i < show:
                print(f"  {os.path.basename(row['path']):30s} -> {CLASS_NAMES[pred]:13s} "
                      f"({probs[pred].item():.2%}) | true: {CLASS_NAMES[row['label']]}")
    print(f"Accuracy on letter '{letter}': {correct/total:.2%} ({correct}/{total})")

# quick check of whether filenames actually encode a usable letter:
print(manifest['letter'].value_counts().head(10))

# Example usage:
# predict_image('/content/dyslexic_dataset/.../some_image.png')
# predict_folder('/content/some_folder_of_images')
# predict_letter('A')


## Notes

- **Why this fixes the earlier problem**: Not-Dyslexic (`normal`) and Dyslexic (`reversal`+`corrected`) now come
  from the *same* dataset/collection pipeline, so the model can't shortcut on file-format or resolution
  artifacts the way it did when Normal came from a completely different downloaded source.
- **Per-class breakdown matters here**: `corrected` (8,029 originally) is much smaller than `reversal` (52,196)
  and `normal` (78,275) — watch its accuracy specifically in step 6; if it's much worse than `reversal`, that's
  a sample-size issue, not a Vision Mamba issue, and more `corrected` examples (or oversampling it specifically)
  would help.
- **Letters**: this dataset's folders don't preserve per-letter identity by default. If filenames don't encode
  a leading letter, `predict_letter` won't be usable — that's expected, not a bug. If you specifically need
  per-letter breakdowns, that requires a version of this dataset (or your own labeling pass) that keeps the
  letter for each sample.
- **IMG_SIZE=64 / PATCH_SIZE=8** balances detail (useful for spotting mirrored/reversed strokes) against SSM
  scan cost. Push to 32/4 for speed on very large runs, or higher if your images are natively larger and you
  want finer detail.
